# Script for preparing data

This script extracts features for MegaDescriptor. **Cell 1** installs deps, mounts Drive, and downloads the models (~2 GB).

In [ ]:
import os
import sys
import subprocess

def in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return os.path.isdir('/content')

def find_repo_root():
    path = os.path.abspath(os.getcwd())
    for _ in range(6):
        if os.path.isdir(os.path.join(path, 'sides_matching')):
            return path
        parent = os.path.dirname(path)
        if parent == path:
            break
        path = parent
    for candidate in ['/content/sides-matching', '/content']:
        if os.path.isdir(os.path.join(candidate, 'sides_matching')):
            return candidate
    return None

def _purge_sides_matching_modules():
    for name in list(sys.modules):
        if name == 'sides_matching' or name.startswith('sides_matching.'):
            del sys.modules[name]

def ensure_repo_root():
    clone_dir = '/content/sides-matching'
    repo_url = 'https://github.com/abui-am/side-matching.git'
    root = find_repo_root()
    if root is None:
        if not in_colab():
            raise RuntimeError(
                'Could not find sides_matching. Open the sides-matching folder '
                'in Cursor and connect the Colab extension from this repo.'
            )
        print('Cloning side-matching repo...')
        subprocess.check_call(['git', 'clone', '--depth', '1', repo_url, clone_dir])
        root = clone_dir
    elif in_colab() and os.path.isdir(os.path.join(root, '.git')):
        print(f'Updating repo at {root} to origin/main...')
        subprocess.check_call(['git', '-C', root, 'fetch', '--depth', '1', 'origin', 'main'])
        subprocess.check_call(['git', '-C', root, 'reset', '--hard', 'origin/main'])
    _purge_sides_matching_modules()
    return root

def pip_install(*packages):
    cmd = [sys.executable, '-m', 'pip', 'install', *packages]
    print('>', ' '.join(cmd))
    subprocess.check_call(cmd)

repo_root = ensure_repo_root()

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

pip_install('wildlife-datasets', 'timm', 'scikit-image')
pip_install('git+https://github.com/WildlifeDatasets/wildlife-tools@main')
if not in_colab():
    pip_install('ipykernel')

DRIVE_DATA = '/content/drive/MyDrive/SeaTurtle'
DATASET_DIRS = ('AmvrakikosTurtles', 'ReunionTurtles', 'ZakynthosTurtles')

def has_datasets(path):
    return all(os.path.isdir(os.path.join(path, name)) for name in DATASET_DIRS)

def find_data_dir():
    if in_colab():
        from google.colab import drive
        drive.mount('/content/drive')
        if has_datasets(DRIVE_DATA):
            return DRIVE_DATA
        raise FileNotFoundError(
            f'Datasets not found at {DRIVE_DATA}. '
            'Put AmvrakikosTurtles, ReunionTurtles, ZakynthosTurtles in Drive Saya > SeaTurtle.'
        )
    local = os.path.join(repo_root, 'data')
    if has_datasets(local):
        return local
    raise FileNotFoundError(f'Datasets not found at {local}')

data_dir = find_data_dir()
print(f'Repo: {repo_root}')
print(f'Data: {data_dir} OK')

import timm
import torch
from wildlife_tools.features import DeepFeatures, AlikedExtractor

model_name = 'hf-hub:BVRA/MegaDescriptor-L-384'
img_size = 384
batch_size = 32
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Device: {device}')
print(f'Downloading {model_name} (~2 GB)...')
model = timm.create_model(model_name, num_classes=0, pretrained=True)
model = model.to(device)
model.eval()
megadescriptor = DeepFeatures(model, batch_size=batch_size, device=device)
print('MegaDescriptor ready.')

print('Downloading Aliked weights...')
aliked = AlikedExtractor(batch_size=batch_size, device=device)
print('Aliked ready.')

> /usr/bin/python3 -m pip install wildlife-datasets timm scikit-image
> /usr/bin/python3 -m pip install git+https://github.com/WildlifeDatasets/wildlife-tools@main
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Repo: /content/sides-matching
Data: /content/drive/MyDrive/SeaTurtle OK


In [ ]:
import os
import sys
from wildlife_datasets import datasets

if 'repo_root' not in globals():
    raise RuntimeError('Run the setup cell (cell 1) first.')
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from sides_matching import get_features, get_transform
from sides_matching import amvrakikos, reunion_green, reunion_hawksbill, zakynthos

root_data = data_dir
root_features = os.path.join(DRIVE_DATA, 'features') if in_colab() else os.path.join(repo_root, 'features')
data = [
    ('Amvrakikos', os.path.join(root_data, 'AmvrakikosTurtles'), amvrakikos),
    ('ReunionGreen', os.path.join(root_data, 'ReunionTurtles'), reunion_green),
    ('ReunionHawksbill', os.path.join(root_data, 'ReunionTurtles'), reunion_hawksbill),
    ('Zakynthos', os.path.join(root_data, 'ZakynthosTurtles'), zakynthos),
]

Data must already be present under `../data` (see `analysis_datasets.ipynb`). We only verify the folders exist — nothing is downloaded here.

In [ ]:
for dataset_class in [datasets.AmvrakikosTurtles, datasets.ReunionTurtles, datasets.ZakynthosTurtles]:
    root = os.path.join(root_data, dataset_class.__name__)
    if not os.path.isdir(root):
        raise FileNotFoundError(f'Missing dataset at {root}. Ensure data/ is synced to Colab.')
    print(f'Using existing data: {root}')

Using existing data: /content/drive/MyDrive/SeaTurtle/AmvrakikosTurtles
Using existing data: /content/drive/MyDrive/SeaTurtle/ReunionTurtles
Using existing data: /content/drive/MyDrive/SeaTurtle/ZakynthosTurtles


Extract features using the models downloaded in cell 1.

In [ ]:
if 'megadescriptor' not in globals():
    raise RuntimeError('Run cell 1 first to download models.')

os.makedirs(root_features, exist_ok=True)
for name, root, dataset_class in data:
    for flip in [True, False]:
        for grayscale in [True, False]:
            transform = get_transform(flip=flip, grayscale=grayscale, img_size=img_size, normalize=True)
            dataset = dataset_class(root, transform=transform, load_label=True)
            file_name = os.path.join(root_features, f'MegaDescriptor_{name}_flip={flip}_grayscale={grayscale}.pickle')
            if not os.path.exists(file_name):
                get_features(file_name, dataset, megadescriptor)
            file_name = os.path.join(root_features, f'Aliked_{name}_flip={flip}_grayscale={grayscale}_{img_size}.pickle')
            if not os.path.exists(file_name):
                get_features(file_name, dataset, aliked)

In [ ]:
Save features to **Drive Saya → SeaTurtle** (works from Cursor Colab extension — no browser download needed).

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.94GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

100%|█████████████████████████████████████████████████████████████████| 7/7 [03:49<00:00, 32.83s/it]


Downloading: "https://github.com/Shiaoming/ALIKED/raw/main/models/aliked-n16.pth" to /root/.cache/torch/hub/checkpoints/aliked-n16.pth


100%|██████████| 2.61M/2.61M [00:00<00:00, 58.0MB/s]
100%|█████████████████████████████████████████████████████████████| 160/160 [00:39<00:00,  4.10it/s]


In [ ]:
import glob
import shutil

if 'root_features' not in globals():
    raise RuntimeError('Run the feature extraction cell first.')

drive_features = os.path.join(DRIVE_DATA, 'features')
os.makedirs(drive_features, exist_ok=True)

for src_dir in {root_features, os.path.join(repo_root, 'features')}:
    if src_dir == drive_features or not os.path.isdir(src_dir):
        continue
    for path in glob.glob(os.path.join(src_dir, '*.pickle')):
        dest = os.path.join(drive_features, os.path.basename(path))
        if not os.path.exists(dest):
            print(f'Copying {os.path.basename(path)}...')
            shutil.copy2(path, dest)

pickles = sorted(glob.glob(os.path.join(drive_features, '*.pickle')))
if not pickles:
    raise FileNotFoundError(f'No feature files in {drive_features}')

total_mb = sum(os.path.getsize(p) for p in pickles) / 1e6
print(f'\n{len(pickles)} files saved to Drive: {drive_features}')
print(f'Total: {total_mb:.0f} MB')

zip_base = os.path.join(DRIVE_DATA, 'features')
zip_path = zip_base + '.zip'
if not os.path.exists(zip_path):
    print(f'\nCreating features.zip on Drive (~{total_mb:.0f} MB, may take a few minutes)...')
    shutil.make_archive(zip_base, 'zip', drive_features)
print(f'Zip: {zip_path} ({os.path.getsize(zip_path) / 1e6:.0f} MB)')
print('\nDownload from: drive.google.com → Drive Saya → SeaTurtle → features.zip')


32 feature files in /content/sides-matching/features
  Aliked_Amvrakikos_flip=False_grayscale=False_384.pickle (145.4 MB)
  Aliked_Amvrakikos_flip=False_grayscale=True_384.pickle (145.4 MB)
  Aliked_Amvrakikos_flip=True_grayscale=False_384.pickle (145.4 MB)
  Aliked_Amvrakikos_flip=True_grayscale=True_384.pickle (145.4 MB)
  Aliked_ReunionGreen_flip=False_grayscale=False_384.pickle (145.4 MB)
  Aliked_ReunionGreen_flip=False_grayscale=True_384.pickle (145.4 MB)
  Aliked_ReunionGreen_flip=True_grayscale=False_384.pickle (145.4 MB)
  Aliked_ReunionGreen_flip=True_grayscale=True_384.pickle (145.4 MB)
  Aliked_ReunionHawksbill_flip=False_grayscale=False_384.pickle (98.9 MB)
  Aliked_ReunionHawksbill_flip=False_grayscale=True_384.pickle (98.9 MB)
  Aliked_ReunionHawksbill_flip=True_grayscale=False_384.pickle (98.9 MB)
  Aliked_ReunionHawksbill_flip=True_grayscale=True_384.pickle (98.9 MB)
  Aliked_Zakynthos_flip=False_grayscale=False_384.pickle (116.3 MB)
  Aliked_Zakynthos_flip=False_grays

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files
zip_path = shutil.make_archive('/content/features', 'zip', root_features)
zip_mb = os.path.getsize(zip_path) / 1e6
print(f'\nDownloading features.zip ({zip_mb:.0f} MB)...')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>